# 1. Cấu hình

Notebook này clone mã nguồn từ repo riêng bằng GitHub Secret; không đưa token vào URL hoặc nội dung cell. Chỉ chạy baseline M3 trên GPU Colab, với dev cases và system prompt đã có trong repo. Trước khi chạy, đẩy commit muốn dùng lên nhánh main; có thể cố định commit bằng biến bên dưới. Outputs lưu vào Google Drive theo mặc định.


In [ ]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/coder-fishing/KidStory.git"
BRANCH = "codex/m3"
COMMIT_TO_RUN = None  # Có thể đặt SHA đã push để khóa chính xác mã nguồn.
REPO_DIR = Path("/content/kids-story")
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/kids-story-runs/m3")
RUNTIME_OUTPUT_ROOT = Path("/content/outputs/m3")


# 2. Truy cập repo riêng

Trong Colab, mở Secrets (biểu tượng chìa khóa), thêm `GITHUB_TOKEN` với quyền đọc repository và bật quyền truy cập cho notebook. Nếu thiếu secret hoặc clone thất bại, cell dừng với hướng dẫn chung; token không được in.


In [ ]:
import os
import subprocess
import tempfile

try:
    from google.colab import userdata
except ImportError as exc:
    raise RuntimeError("Cell này cần chạy trong Google Colab.") from exc

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception as exc:
    raise RuntimeError("Không đọc được secret GITHUB_TOKEN. Hãy thêm secret và cấp quyền cho notebook.") from exc
if not github_token:
    raise RuntimeError("Thiếu secret GITHUB_TOKEN. Hãy thêm secret có quyền đọc repository.")
if REPO_DIR.exists():
    raise RuntimeError(f"Thư mục {REPO_DIR} đã tồn tại; hãy dùng runtime mới để tránh dùng nhầm mã nguồn.")

with tempfile.TemporaryDirectory() as askpass_dir:
    askpass_path = Path(askpass_dir) / "git-askpass.sh"
    askpass_path.write_text(
        "#!/bin/sh\ncase \"$1\" in\n  *Username*) printf '%s\\n' 'x-access-token' ;;\n  *Password*) printf '%s\\n' \"$GITHUB_TOKEN\" ;;\nesac\n",
        encoding="utf-8",
    )
    askpass_path.chmod(0o700)
    git_env = os.environ.copy()
    git_env["GIT_ASKPASS"] = str(askpass_path)
    git_env["GITHUB_TOKEN"] = github_token
    git_env["GIT_TERMINAL_PROMPT"] = "0"
    try:
        clone = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            env=git_env, capture_output=True, text=True,
        )
    finally:
        git_env.pop("GITHUB_TOKEN", None)
        del git_env
github_token = None
del github_token
if clone.returncode:
    raise RuntimeError("Clone thất bại. Kiểm tra URL repo, quyền đọc của token và nhánh main; thông tin xác thực đã được ẩn.")

if COMMIT_TO_RUN:
    checkout = subprocess.run(["git", "checkout", "--detach", COMMIT_TO_RUN], cwd=REPO_DIR, capture_output=True, text=True)
    if checkout.returncode:
        raise RuntimeError("Không checkout được COMMIT_TO_RUN; xác nhận SHA đã có trên remote.")
source_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True,
).stdout.strip()
source_status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPO_DIR, check=True, capture_output=True, text=True,
).stdout
if source_status.strip():
    raise RuntimeError("Repo clone không sạch; dừng để tránh chạy mã nguồn đã bị sửa.")
print(f"Đã lấy mã nguồn commit {source_commit}; working tree sạch.")


# 3. Kiểm tra GPU Colab

Cell này kiểm tra runtime trước khi đọc dev cases hoặc tải model. Nếu không có GPU CUDA, dừng tại đây; không tự chuyển sang CPU.


In [ ]:
import platform
try:
    import torch
except ImportError as exc:
    raise RuntimeError('Colab PyTorch is missing; select a standard Colab Python runtime.') from exc

torch_before_install = torch.__version__
runtime_info = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
}
if not runtime_info["cuda_available"]:
    raise RuntimeError("Không phát hiện GPU CUDA. Trong Colab chọn Runtime > Change runtime type > GPU rồi chạy lại.")
runtime_info["gpu_name"] = torch.cuda.get_device_name(0)
runtime_info["gpu_memory_bytes"] = int(torch.cuda.get_device_properties(0).total_memory)
print(runtime_info)


# 4. Cài dependencies

PyTorch do Colab cung cấp được giữ nguyên. Chỉ cài các dependency inference đã pin; Unsloth thuộc M4, không dùng trong notebook M3 này.


In [ ]:
%pip install -r /content/kids-story/requirements/m3-colab.lock.txt


In [ ]:
import importlib.metadata

expected_versions = {"transformers": "4.57.3", "accelerate": "1.12.0"}
actual_versions = {}
for name in expected_versions:
    try:
        actual_versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError(f'Required package missing: {name}. Re-run the dependency installation cell.') from exc
if actual_versions != expected_versions:
    raise RuntimeError(f"Phiên bản dependency không khớp lock: {actual_versions}")
if torch.__version__ != torch_before_install:
    raise RuntimeError(f"PyTorch changed during dependency install: {torch_before_install} -> {torch.__version__}")
print({"installed": actual_versions, "torch_kept_from_colab": torch.__version__})


# 5. Chạy baseline

Mặc định notebook mount Google Drive bằng giao diện Colab để giữ artifact sau khi runtime kết thúc. Nếu tắt tùy chọn đó, output chỉ nằm trong runtime tạm thời. Mỗi lần chạy tạo thư mục mới theo UTC; runner từ chối ghi đè.


In [ ]:
import hashlib
import json
import sys
from datetime import datetime, timezone

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_parent = DRIVE_OUTPUT_ROOT
else:
    output_parent = RUNTIME_OUTPUT_ROOT
output_parent.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("qwen25_3b_%Y%m%dT%H%M%SZ")
output_dir = output_parent / run_id

sys.path.insert(0, str(REPO_DIR / "src"))
from kids_story.baseline import MODEL_ID, MODEL_REVISION, run_baseline
if MODEL_ID != 'Qwen/Qwen2.5-3B-Instruct' or MODEL_REVISION != 'aa8e72537993ba99e69dfaafa59ed015b17504d1':
    raise RuntimeError('Pinned model ID/revision do not match the approved M3 specification.')

input_path = REPO_DIR / "data/evaluation/dev_cases.jsonl"
system_prompt_path = REPO_DIR / "prompts/story_system_v1.txt"
result = run_baseline(
    input_path,
    system_prompt_path,
    output_dir,
    source_commit=source_commit,
)

complete_path = output_dir / "COMPLETE.json"
completion = json.loads(complete_path.read_text(encoding="utf-8"))
for filename, expected_sha256 in completion["files"].items():
    actual_sha256 = hashlib.sha256((output_dir / filename).read_bytes()).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f"Hash artifact không khớp: {filename}")
print({"run_id": run_id, "status": result["status"], "case_count": result["total_cases"], "artifact_dir": str(output_dir), "hashes_verified": True})
if not SAVE_TO_DRIVE:
    print("CẢNH BÁO: artifacts chỉ nằm trong runtime Colab tạm thời; hãy tải/copy chúng trước khi kết thúc phiên.")


## Giới hạn lần chạy

Chạy thành công chỉ tạo artifact baseline trên dev; không chứng minh chất lượng, an toàn hoặc cải thiện so với fine-tune. Không dùng golden draft để chỉnh prompt/chọn model. M2 vẫn có các cổng duyệt riêng đang chờ.
